# Spur Review: six requirement repair lab

Each gate has a separate experiment: R1 named Deno launch; R2 named Python publication; R3 named Deno consumption; R4 real DuckPGQ execution; R5 native app manifest/doctor; R6 live review collectors. This lab separates runtime defects, deliberate unsupported contracts and missing app integration. It never silently switches a named cell to standalone.

R4 begins by reproducing the prior failure, then activates the DuckDB 2.0 extension grammar on the same connection. R1–R3 are source-grounded against the current build; configuring a profile is not evidence of successful execution. App and collector experiments remain independent so runtime gaps do not hide otherwise testable work.

In [2]:
import os,sys,json,hashlib
from pathlib import Path
import duckdb
assert duckdb.__version__=='2.0.0'
review_fixture=Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-spur-review-ux/design-data.json')
review_data=json.loads(review_fixture.read_text())
assert hashlib.sha256(review_fixture.read_bytes()).hexdigest()=='b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2'
engine=duckdb.connect(config={'allow_unsigned_extensions':True})
binaries=json.loads(Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-17-review-extensions-poc/extension-binaries.json').read_text())
for binary in binaries:
 p=Path(binary['path']); assert hashlib.sha256(p.read_bytes()).hexdigest()==binary['sha256']
 engine.execute("LOAD '"+str(p).replace("'","''")+"'")
print(json.dumps({'profile':'review-engine','executable':sys.executable,'pid':os.getpid(),'duckdb':duckdb.__version__,'cwd':os.getcwd(),'loaded':engine.sql("SELECT extension_name,loaded FROM duckdb_extensions() WHERE extension_name IN ('spur_graph','duck_tails','duckpgq') ORDER BY extension_name").fetchall(),'active_grammar_extensions':engine.sql("SELECT current_setting('active_grammar_extensions')").fetchall()[0][0]},indent=2))

{
  "profile": "review-engine",
  "executable": "/Users/kevintruong/.spur/jupyter/venv-duckdb-2.0/bin/python",
  "pid": 62136,
  "duckdb": "2.0.0",
  "cwd": "/private/var/folders/mv/vtzfmys92yl9rwcsgf9n1x780000gn/T/spur-review-named-kibzfbbp",
  "loaded": [
    [
      "duck_tails",
      true
    ],
    [
      "duckpgq",
      true
    ],
    [
      "spur_graph",
      true
    ]
  ],
  "active_grammar_extensions": []
}


In [3]:
# R4: same connection, reversible grammar experiment, then real captured-context traversal.
engine.execute('CREATE OR REPLACE TABLE review_nodes(id BIGINT PRIMARY KEY,anchor VARCHAR,state VARCHAR,evidence VARCHAR)')
all_nodes=[(ctx+':'+n['id'],n) for ctx,g in review_data['contexts'].items() for n in g['nodes']]
node_id={key:i for i,(key,n) in enumerate(all_nodes)}
engine.executemany('INSERT INTO review_nodes VALUES (?,?,?,?)',[(node_id[key],key,n['state'],n['evidence']) for key,n in all_nodes])
engine.execute('CREATE OR REPLACE TABLE review_edges(src BIGINT,dst BIGINT,kind VARCHAR,evidence VARCHAR)')
edge_rows=[(node_id[ctx+':'+e['a']],node_id[ctx+':'+e['b']],e['kind'],e['evidence']) for ctx,g in review_data['contexts'].items() for e in g['edges']]
engine.executemany('INSERT INTO review_edges VALUES (?,?,?,?)',edge_rows)
engine.execute("CREATE OR REPLACE TABLE syntax_edges AS SELECT * FROM review_edges WHERE kind='syntax'")
grammar_probe={'before':engine.sql("SELECT current_setting('active_grammar_extensions')").fetchall()[0][0]}
def create_graph(name,edges):
 engine.execute(f"""CREATE OR REPLACE PROPERTY GRAPH {name}
 VERTEX TABLES (review_nodes LABEL symbol)
 EDGE TABLES ({edges} SOURCE KEY(src) REFERENCES review_nodes(id)
 DESTINATION KEY(dst) REFERENCES review_nodes(id) LABEL relates)""")
try:
 create_graph('red_graph','review_edges')
 grammar_probe['red']='unexpected_success'
except duckdb.ParserException as exc:
 grammar_probe['red']=str(exc)
assert 'PROPERTY' in grammar_probe['red']
engine.execute("SET active_grammar_extensions = ['duckpgq']")
create_graph('review_context','review_edges')
create_graph('review_syntax','syntax_edges')
path_query="""SELECT * FROM GRAPH_TABLE({graph}
 MATCH p = ANY SHORTEST (a:symbol)-[e:relates]->{{1,2}}(b:symbol)
 WHERE b.anchor='incoming:internal'
 COLUMNS (a.anchor AS source,a.state AS source_state,path_length(p) AS hops))
 ORDER BY hops,source"""
pgq_context=engine.sql(path_query.format(graph='review_context')).fetchall()
pgq_syntax=engine.sql(path_query.format(graph='review_syntax')).fetchall()
sql_query="""WITH RECURSIVE allowed AS (SELECT * FROM review_edges {filter}),
walk(source,dst,hops) AS (SELECT src,dst,1 FROM allowed UNION ALL
SELECT w.source,e.dst,w.hops+1 FROM walk w JOIN allowed e ON w.dst=e.src WHERE w.hops<2)
SELECT n.anchor,n.state,w.hops FROM walk w
JOIN review_nodes n ON n.id=w.source JOIN review_nodes b ON b.id=w.dst
WHERE b.anchor='incoming:internal' ORDER BY w.hops,n.anchor"""
assert pgq_context==engine.sql(sql_query.format(filter='')).fetchall()
assert pgq_syntax==engine.sql(sql_query.format(filter="WHERE kind='syntax'")).fetchall()
assert len(pgq_context)==3 and len(pgq_syntax)==1
grammar_probe.update(context_paths=pgq_context,syntax_paths=pgq_syntax,recursive_sql_parity=True)
engine.execute("SET active_grammar_extensions = []")
try:
 engine.sql(path_query.format(graph='review_context')).fetchall()
 grammar_probe['reverse']='unexpected_success'
except duckdb.ParserException as exc:
 grammar_probe['reverse']=str(exc)
assert grammar_probe['reverse']!='unexpected_success'
engine.execute("SET active_grammar_extensions = ['duckpgq']")
grammar_probe['after']=engine.sql("SELECT current_setting('active_grammar_extensions')").fetchall()[0][0]
print(json.dumps(grammar_probe,indent=2))

{
  "before": [],
  "red": "Parser Error: syntax error at or near \"PROPERTY\"\n\nLINE 1: CREATE OR REPLACE PROPERTY GRAPH red_graph\n                          ^^^^^^^^",
  "context_paths": [
    [
      "incoming:wrapper",
      "Unchanged body",
      1
    ],
    [
      "incoming:client",
      "Unchanged",
      2
    ],
    [
      "incoming:extractor",
      "Unchanged",
      2
    ]
  ],
  "syntax_paths": [
    [
      "incoming:wrapper",
      "Unchanged body",
      1
    ]
  ],
  "recursive_sql_parity": true,
  "reverse": "Parser Error: syntax error at or near \"MATCH\"\n\nLINE 2:  MATCH p = ANY SHORTEST (a:symbol)-[e:relates]->{1,2}(b:symbol...\n         ^^^^^",
  "after": [
    "duckpgq"
  ]
}


In [4]:
# R2: observe the deliberately disabled named Python API instead of patching it.
import importlib
port_stub=importlib.import_module('spur')
python_port_probe={'module':getattr(port_stub,'__file__',None),'global_before_import':'spur' in globals(),'operations':[]}
for method,args in [('put',('review_probe',[{'run_head':review_data['head']}])) ,('get',('review_probe',))]:
 try:
  getattr(port_stub,method)(*args)
  python_port_probe['operations'].append({'method':method,'unexpected_success':True})
 except Exception as exc:
  python_port_probe['operations'].append({'method':method,'type':type(exc).__name__,'message':str(exc)})
assert all('ports_unavailable' in r.get('message','') for r in python_port_probe['operations'])
print(json.dumps(python_port_probe,indent=2))

{
  "module": null,
  "global_before_import": false,
  "operations": [
    {
      "method": "put",
      "type": "RuntimeError",
      "message": "ports_unavailable: named Python ports require a qualified host publication bridge"
    },
    {
      "method": "get",
      "type": "RuntimeError",
      "message": "ports_unavailable: named Python ports require a qualified host publication bridge"
    }
  ]
}


In [5]:
# R2 origin experiment A: no port calls and no mutation of publication guards.
import asyncio,threading
origin_kernel=get_ipython().kernel
origin_a=origin_kernel.get_parent()['header']['msg_id']
origin_records={}
origin_async_gate=asyncio.Event()
origin_thread_gate=threading.Event()
async def origin_async_worker():
 await origin_async_gate.wait()
 origin_records['delayed_async']=origin_kernel.get_parent()['header']['msg_id']
def origin_thread_worker():
 origin_thread_gate.wait()
 origin_records['plain_thread']=origin_kernel.get_parent()['header']['msg_id']
origin_async_task=asyncio.create_task(origin_async_worker())
origin_thread=threading.Thread(target=origin_thread_worker,daemon=True)
origin_thread.start()
print(json.dumps({'phase':'A','origin':origin_a,'workers':'waiting; run phase B next'}))

{"phase": "A", "origin": "6262e9b3-565b-4290-ac5d-6ded24e7b831", "workers": "waiting; run phase B next"}


In [6]:
# R2 origin experiment B: old background work observes its execution context.
origin_b=origin_kernel.get_parent()['header']['msg_id']
origin_async_gate.set(); origin_thread_gate.set()
await origin_async_task
origin_thread.join()
origin_observation={'phase_a':origin_a,'phase_b':origin_b,**origin_records,
 'async_preserved_A':origin_records['delayed_async']==origin_a,
 'plain_thread_reports_B':origin_records['plain_thread']==origin_b,
 'interpretation':'An unbound thread must not acquire publication authority merely from the latest parent message.'}
print(json.dumps(origin_observation,indent=2))

{
  "phase_a": "6262e9b3-565b-4290-ac5d-6ded24e7b831",
  "phase_b": "1946b01c-0716-47dc-8d14-0ec72494a0e6",
  "plain_thread": "1946b01c-0716-47dc-8d14-0ec72494a0e6",
  "delayed_async": "6262e9b3-565b-4290-ac5d-6ded24e7b831",
  "async_preserved_A": true,
  "plain_thread_reports_B": true,
  "interpretation": "An unbound thread must not acquire publication authority merely from the latest parent message."
}


## R6 collector plan

Reuse the exact existing `github-pat` connection definition to attach its read-only query surface to this lab; keep its credential reference in the host. Run a narrow PR inbox query, then collect pinned Git tree/diff and DuckTails history through the named DuckDB 2.0 engine. A live inbox response is not yet a coherent review run: base/head and repository identities must still be joined before publication.

## R5 app plan

A separate `vanilla-widget` package will carry the existing captured review UI and a valid app manifest. It declares active output scripts for its existing local interaction code and no external workspace or server access. Doctor and a rendered control qualify packaging/display only. Named Deno frontend execution and port-driven refresh remain separate requirements; the scaffold cannot qualify them.

In [5]:
# SPUR datasource setup cell v1
# This cell is managed by SPUR and runs when connections change.
# SPUR DuckDB bootstrap
import duckdb
import sys
_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
# Shared by notebook setup and classic SQL cells. Preserve the generic path for
# older bundles; new bundles carry exact native ABI artifacts per engine.
from pathlib import Path as _SpurExtensionPath

_spur_extension = _SpurExtensionPath(_SPUR_DUCKDB_EXTENSION_PATH)
_spur_versioned_extension = (
    _spur_extension.parent
    / ("duckdb-v" + duckdb.__version__)
    / _spur_extension.name
)
if _spur_versioned_extension.is_file():
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)

_SPUR_DUCKDB_GATEWAY_COMPATIBLE = duckdb.__version__ in ("1.4.4", "1.5.5")

if duckdb.__version__ == "2.0.0":
    from platform import machine as _spur_machine

    # Never leave the generic legacy path available to an exact-2.0 caller,
    # including explicit datasource setup when the artifact is missing.
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)
    _spur_gateway_error = ""
    if sys.platform != "darwin" or _spur_machine().lower() not in ("arm64", "aarch64"):
        _spur_gateway_error = "the 2.0 gateway is qualified only on macOS ARM64"
    else:
        try:
            # DuckDB's footer is eight 32-byte fields and a 256-byte signature,
            # preceded by its 22-byte metadata marker. Check final signed bytes.
            with _spur_versioned_extension.open("rb") as _spur_artifact:
                _spur_artifact.seek(-534, 2)
                _spur_footer = _spur_artifact.read(534)
            _spur_fields = [
                _spur_footer[22 + i * 32:22 + (i + 1) * 32].rstrip(b"\x00")
                for i in range(8)
            ]
            if (
                _spur_footer[:22] != b"\x00\x93\x04\x10duckdb_signature\x80\x04"
                or _spur_fields[3] != b"C_STRUCT_UNSTABLE"
                or _spur_fields[5:] != [b"v2.0.0", b"osx_arm64", b"4"]
            ):
                _spur_gateway_error = "the artifact footer does not match DuckDB 2.0.0 on macOS ARM64"
        except OSError:
            _spur_gateway_error = "the exact 2.0.0 artifact is missing or unreadable"
    _SPUR_DUCKDB_GATEWAY_COMPATIBLE = not _spur_gateway_error
    if _spur_gateway_error:
        print(
            f"SPUR: {_spur_gateway_error}. Install or rebuild Jute with the matching "
            f"gateway artifact ({_SPUR_DUCKDB_EXTENSION_PATH}), then restart the notebook "
            "kernel. Pure SQL remains available.",
            file=sys.stderr,
        )

if (
    sys.platform in ("linux", "darwin")
    and _SPUR_DUCKDB_GATEWAY_COMPATIBLE
    and "_SPUR_DUCKDB_HOST_SYMBOLS" not in globals()
):
    import ctypes as _spur_ctypes
    import _duckdb as _spur_host_duckdb

    # Promote the already-loaded Python engine's symbols. This opens the same
    # module handle; it neither imports another engine nor replaces connections.
    _SPUR_DUCKDB_HOST_SYMBOLS = _spur_ctypes.CDLL(
        _spur_host_duckdb.__file__, mode=_spur_ctypes.RTLD_GLOBAL
    )

_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true",
                 "autoload_known_extensions": "false",
                 "autoinstall_known_extensions": "false"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
# Embedded helper source contains no token, credential path or capability lease.
# The approved local launcher passes a private inherited socket descriptor, not
# executable source or a notebook/widget message. Same-kernel code is trusted.
def _spur_managed_connection(db):
    """Only the host's selected connection is subject to managed policy."""
    import types
    connection = db.default_connection() if isinstance(db, types.ModuleType) else db
    if connection is not globals().get("_SPUR_DUCKDB_CONNECTION"):
        return None
    try:
        connection.sql("SET autoload_known_extensions = false")
        connection.sql("SET autoinstall_known_extensions = false")
    except Exception:
        return None  # existing locked/custom connections have an explicit boundary
    return connection


# Required SpurLab baseline: JSON/parquet are statically linked in the managed
# wheel; httpfs supplies remote files and REST gateway dependencies; spur_rest is
# the separately packaged, host-verified gateway (never a repository download).
# Remote datasource attach code explicitly installs/loads its provider adapter.
# Spatial and Jev are optional. No browsing/query path calls operate("install").
# Fixed metadata query only. Identifiers and descriptions remain inert JSON; no
# catalog value is interpolated as SQL. The limits bound the shell reply too.
def _spur_function_catalog(db):
    import json
    limit = 10000
    rows = db.sql("SELECT database_name, schema_name, function_name, function_type, "
                  "parameters, parameter_types, varargs, return_type, description, examples, function_oid "
                  "FROM duckdb_functions() ORDER BY database_name, schema_name, function_name, "
                  "function_type, parameter_types, varargs LIMIT 10001").fetchall()
    functions, size = [], 0
    complete = len(rows) <= limit
    for row in rows[:limit]:
        value = dict(zip(("database_name", "schema_name", "name", "function_type", "parameters",
                          "parameter_types", "varargs", "return_type", "description", "examples", "function_oid"), row))
        value["ownership"] = "unknown"
        encoded = json.dumps(value, ensure_ascii=True)
        # Oversized metadata is omitted, with incomplete coverage made explicit.
        if len(encoded) > 65536 or size + len(encoded) > 4 * 1024 * 1024:
            complete = False
            continue
        functions.append(value)
        size += len(encoded)
    return {"status": "complete" if complete else "incomplete", "functions": functions,
            "truncated": not complete}


def _spur_function_identity(value):
    return (value["database_name"], value["schema_name"], value["name"],
            value["function_type"], tuple(value["parameter_types"]), value["varargs"])


def _spur_snapshot_function_registration(db):
    try:
        snapshot = _spur_function_catalog(db)
        if snapshot["status"] == "complete" and snapshot.get("truncated") is False:
            return snapshot
    except Exception:
        pass  # Optional metadata must not prevent an authorized extension load.
    return None


def _spur_record_function_ownership(db, extension, before):
    """Retain registrations observed around a verified load on this connection."""
    if before is None:
        return
    try:
        if before["status"] != "complete" or before.get("truncated") is not False:
            return
        after = _spur_snapshot_function_registration(db)
        if after is None:
            return
        def key(row):
            return (*_spur_function_identity(row), row.get("return_type"), row["function_oid"])
        previous = {key(row) for row in before["functions"]}
        registrations = globals().setdefault("_SPUR_DUCKDB_FUNCTION_OWNERSHIP", {})
        retained = registrations.get(id(db))
        owners = dict(retained[1]) if retained is not None and retained[0] is db else {}
        additions = [key(row) for row in after["functions"]
                     if row["function_oid"] is not None and key(row) not in previous]
        for identity in additions:
            owners[identity] = set(owners.get(identity, ())) | {extension}
        registrations[id(db)] = (db, owners)
    except Exception:
        pass  # Incomplete/failed evidence stays unattributed.


class _SpurDuckDBRuntime:
    BASELINE = ("json", "parquet", "httpfs", "spur_rest")
    REPOSITORIES = {
        # DuckDB's built-in installer handles HTTP without httpfs; HTTPS itself
        # requires httpfs. Use the official signed distribution bootstrap, never
        # temporarily re-enable autoload to download its own dependency.
        "httpfs": "http://extensions.duckdb.org",
        "spatial": "http://extensions.duckdb.org",
        "jev": "http://community-extensions.duckdb.org",
    }

    def __init__(self, db, platform):
        self.db, self.platform = db, platform
        self.entries = {}
        self.inventories = {}  # inert metadata; never activation authority
        self.verified = {}  # at most one private verified copy per curated entry
        self.loaded = set()  # artifact verification is not selected-connection load

    def _entry(self, name):
        return self.entries.setdefault(name, {
            "name": name, "installed": None,
            "desired_enabled": name in self.BASELINE, "loaded": None,
            "provider_health": "unknown", "restart_required": False,
            "activation_transport_restart": False,
            "stage": "idle", "reason": None, "policy": "unknown",
            "artifact_digest": None, "signature_verified": False,
        })

    def _observe(self, db, name):
        rows = db.sql(
            "SELECT loaded, installed, install_path, installed_from, install_mode "
            "FROM duckdb_extensions() WHERE extension_name = ?", params=[name]
        ).fetchall()
        return rows[0] if rows else None

    def _refresh(self, entry):
        row = self._observe(self.db, entry["name"])
        entry["loaded"] = bool(row and row[0])
        entry["installed"] = bool(row and row[1])
        return row

    def _artifact(self, name, install, cancelled):
        # Verification happens in a separate database with signature and ABI
        # checks ON, even when the selected gateway connection allows unsigned
        # bundled code. A repository label or hash alone is not signing evidence.
        import duckdb
        import hashlib
        from pathlib import Path
        import shutil
        import tempfile
        version = self.db.sql("SELECT version()").fetchall()[0][0]
        config = {"allow_unsigned_extensions": "false",
                  "allow_community_extensions": "true" if name == "jev" else "false",
                  "allow_extensions_metadata_mismatch": "false",
                  "autoload_known_extensions": "false",
                  "autoinstall_known_extensions": "false"}
        if version == "v2.0.0":
            directories = self.db.sql("SELECT current_setting('extension_directories')").fetchall()[0][0]
            if directories:
                config["extension_directories"] = directories
        else:
            directory = self.db.sql("SELECT current_setting('extension_directory')").fetchall()[0][0]
            config["home_directory"] = self.db.sql("SELECT current_setting('home_directory')").fetchall()[0][0]
            if directory:
                config["extension_directory"] = directory
        entry = self._entry(name)
        entry["signature_verified"] = False
        with duckdb.connect(config=config) as verifier:
            if (version not in ("v1.5.5", "v2.0.0")
                    or verifier.sql("SELECT version()").fetchall()[0][0] != version
                    or verifier.sql("PRAGMA platform").fetchall()[0][0] != self.platform):
                raise ValueError("runtime_mismatch")
            entry["stage"] = "installing" if install else "verifying"
            row = self._observe(verifier, name)
            if install and (row is None or not row[1]):
                # Install only when absent. DuckDB rejects INSTALL across HTTP/
                # HTTPS origin spellings even for the same official repository.
                # Cached artifacts follow the same origin/signature/ABI checks
                # below; never FORCE/UPDATE or silently replace rejected bytes.
                verifier.install_extension(name, repository_url=self.REPOSITORIES[name])
                row = self._observe(verifier, name)
            if row is None or not row[1]:
                raise ValueError("not_installed")
            entry["installed"] = True
            if cancelled():
                raise InterruptedError()
            entry["stage"] = "verifying"
            repository = self.REPOSITORIES[name]
            aliases = (repository, repository.replace("http:", "https:"),
                       "community" if name == "jev" else "core")
            if row[3] not in aliases or row[4] != "REPOSITORY":
                raise ValueError("repository_mismatch")
            # Freeze the bytes in a private directory before the verifier loads
            # them. Load those same bytes into the selected connection, never a
            # mutable cache alias. Same-user/kernel arbitrary code is trusted.
            private = tempfile.TemporaryDirectory(prefix="spur-duckdb-verified-")
            try:
                path = Path(private.name) / (name + ".duckdb_extension")
                shutil.copyfile(row[2], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                try:
                    before = _spur_function_catalog(verifier)
                except Exception:
                    before = None
                verifier.load_extension(str(path))  # engine checks signature/ABI
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError("artifact_changed")
                try:
                    after = _spur_function_catalog(verifier)
                    if before is None:
                        raise ValueError("inventory_failed")
                    previous_rows = {_spur_function_identity(f): f for f in before["functions"]}
                    functions = [f for f in after["functions"]
                                 if previous_rows.get(_spur_function_identity(f)) != f]
                    complete = before["status"] == after["status"] == "complete"
                    inventory = {"status": "complete" if complete else "incomplete",
                                 "functions": functions, "truncated": not complete}
                except Exception:
                    inventory = {"status": "failed", "functions": [], "truncated": False}
                inventory.update(artifact_digest=digest, provenance="isolated_signed_verifier_load_delta")
                self.inventories[name] = inventory
            except BaseException:
                private.cleanup()
                raise
            previous = self.verified.get(name)
            if previous is not None:
                previous.cleanup()
            self.verified[name] = private
            return str(path), digest

    def operate(self, name, action, cancelled=lambda: False):
        """Explicit, nonsecret job; called only on the kernel shell queue.

        Cancellation is cooperative at stages. It never interrupts a user query;
        installed but not loaded artifacts remain available for an explicit retry.
        There is no UNLOAD in DuckDB: disable reports loaded + restart_required.
        """
        entry = self._entry(name)
        entry["stage"], entry["reason"] = "resolving", None
        if action == "disable" and name not in self.BASELINE:
            entry["desired_enabled"] = False
            if name == "jev":
                _spur_close_jev_channel()
        try:
            if name not in (*self.BASELINE, "spatial", "jev"):
                raise ValueError("unsupported_extension")
            if self.db is None:
                raise ValueError("no_connection")
            if self.db is not globals().get("_SPUR_DUCKDB_CONNECTION"):
                entry["policy"] = "unenforceable"
                raise ValueError("unmanaged_connection")
            if (self.db.sql("SELECT version()").fetchall()[0][0] not in ("v1.5.5", "v2.0.0")
                    or self.db.sql("PRAGMA platform").fetchall()[0][0] != self.platform):
                raise ValueError("runtime_mismatch")
            if _spur_managed_connection(self.db) is None:
                entry["policy"] = "unenforceable"
                raise ValueError("policy_unenforceable")
            entry["policy"] = "enforced"
            self._refresh(entry)
            if action == "disable":
                if name in self.BASELINE:
                    raise ValueError("required_baseline")
                entry["desired_enabled"] = False
                entry["restart_required"] = entry["loaded"] or name == "jev"
                entry["activation_transport_restart"] = name == "jev"
                entry["reason"] = "restart_required" if entry["restart_required"] else None
                entry["stage"] = "succeeded"
                return dict(entry)
            if cancelled():
                raise InterruptedError()
            if action == "invalidate_credential_channel" and name == "jev":
                _spur_close_jev_channel()
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
            elif action == "enable":
                entry["desired_enabled"] = True
                if name == "jev" and not self._channel_ready():
                    entry["restart_required"] = True
                    entry["activation_transport_restart"] = True
                    entry["reason"] = self._channel_reason()
            elif action in ("install", "revalidate", "load", "prepare_verification"):
                if action == "prepare_verification" and (name not in self.REPOSITORIES or not entry["installed"]
                        or not entry["signature_verified"] or not entry["artifact_digest"]):
                    raise ValueError("not_installed")
                if action == "load" and not entry["desired_enabled"]:
                    raise ValueError("setup_required_without_autoload")
                if name in ("json", "parquet"):
                    row = self._observe(self.db, name)
                    if not row or row[4] != "STATICALLY_LINKED" or not row[0]:
                        raise ValueError("baseline_unavailable")
                elif name == "spur_rest":
                    # Owned by the bundled artifact installer/bootstrap. Never
                    # weaken signature policy or download a substitute here.
                    if not entry["loaded"]:
                        raise ValueError("baseline_unavailable")
                else:
                    if entry["loaded"] and name == "httpfs" and name not in self.loaded:
                        # Connection setup may have loaded the same signed baseline
                        # first. Accept only its exact-connection frozen-byte proof.
                        from pathlib import Path
                        import hashlib
                        proof = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {}).get((id(self.db), name))
                        if proof and proof[0] is self.db and proof[1] == "core":
                            try:
                                digest = hashlib.sha256((Path(proof[2].name) / "httpfs.duckdb_extension").read_bytes()).digest()
                                if digest == proof[3]:
                                    self.verified[name] = proof[2]
                                    self.loaded.add(name)
                                    entry.update(signature_verified=True, artifact_digest=digest.hex(),
                                                 restart_required=False, activation_transport_restart=False)
                            except OSError:
                                pass
                    if entry["loaded"] and name not in self.loaded:
                        entry["signature_verified"] = False
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("preexisting_load_unverified")
                    if entry["loaded"] and action in ("load", "prepare_verification"):
                        # Explicit retry may reacquire discovery in the same isolated
                        # signature verifier; it must not change selected enablement.
                        if action == "prepare_verification" and self.inventories.get(name, {}).get("status") != "complete":
                            loaded_digest = entry["artifact_digest"]
                            _, refreshed_digest = self._artifact(name, False, cancelled)
                            if refreshed_digest != loaded_digest:
                                entry["restart_required"] = True
                                entry["activation_transport_restart"] = False
                                raise ValueError("artifact_changed")
                            entry["signature_verified"] = True
                        if name == "jev" and action == "load" and not self._channel_ready():
                            entry["restart_required"] = True
                            entry["activation_transport_restart"] = True
                            entry["reason"] = self._channel_reason()
                        if entry["restart_required"] and entry["activation_transport_restart"]:
                            entry["reason"] = self._channel_reason()
                        entry["stage"] = "succeeded"
                        return dict(entry)
                    if entry["loaded"]:
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("restart_required")
                    path, digest = self._artifact(name, action == "install", cancelled)
                    entry["installed"] = True
                    entry["artifact_digest"] = digest
                    entry["signature_verified"] = True
                    if cancelled():
                        raise InterruptedError()
                    if action in ("load", "prepare_verification") and not entry["loaded"]:
                        entry["stage"] = "loading"
                        before = _spur_snapshot_function_registration(self.db)
                        self.db.load_extension(path)
                        self._refresh(entry)
                        if not entry["loaded"]:
                            raise ValueError("load_failed")
                        self.loaded.add(name)
                        _spur_record_function_ownership(self.db, name, before)
                    if name == "jev" and action != "prepare_verification" and not self._channel_ready():
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = True
                        entry["reason"] = self._channel_reason()
            elif action != "inspect":
                raise ValueError("unsupported_operation")
            if entry["restart_required"] and entry["activation_transport_restart"]:
                entry["reason"] = self._channel_reason()
            entry["stage"] = "succeeded"
        except (KeyboardInterrupt, InterruptedError):
            entry["reason"] = "cancelled"
        except ValueError as error:
            # Only our fixed reasons are exposed; never native exception text.
            allowed = {"runtime_mismatch", "unsupported_extension", "unmanaged_connection",
                       "required_baseline", "setup_required_without_autoload", "not_installed",
                       "repository_mismatch", "artifact_changed", "baseline_unavailable",
                       "preexisting_load_unverified", "load_failed", "unsupported_operation", "no_connection",
                       "policy_unenforceable", "restart_required"}
            entry["reason"] = str(error) if str(error) in allowed else "verification_failed"
        except Exception:
            entry["reason"] = {"installing": "install_failed", "loading": "load_failed",
                               "verifying": "verification_failed"}.get(entry["stage"], "policy_failed")
        return dict(entry)

    @staticmethod
    def _channel_ready():
        import os
        if os.name != "posix":
            return False
        try:
            channel = _spur_jev_channel()
            return channel is not None and channel.available()
        except Exception:
            return False

    @staticmethod
    def _channel_reason():
        import os
        return "restart_required" if os.name == "posix" else "credential_transport_unavailable"

    def close(self):
        """Release verified temporary copies after closing this connection."""
        for directory in self.verified.values():
            directory.cleanup()
        self.verified.clear()


def _spur_duckdb_job(platform, name, action, cancel_path=None):
    """Entry point for typed host jobs; no caller SQL and no credential payload."""
    from pathlib import Path
    global _SPUR_DUCKDB_RUNTIME
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is None or current.db is not db or current.platform != platform:
        current = _SPUR_DUCKDB_RUNTIME = _SpurDuckDBRuntime(db, platform)
    return current.operate(name, action, lambda: bool(cancel_path and Path(cancel_path).exists()))


def _spur_duckdb_baseline():
    """Load approved cached baseline only; missing artifacts need explicit setup.

    Used by both root bootstrap and the library's shared SQL session helper.
    The bundled gateway loader stays with its existing host-owned installer.
    """
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    if db is None or db.sql("SELECT version()").fetchall()[0][0] not in ("v1.5.5", "v2.0.0"):
        return
    platform = db.sql("PRAGMA platform").fetchall()[0][0]
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is not None and current.db is not db:
        current = None
    for name in ("json", "parquet", "httpfs"):
        if current is None or not current._entry(name)["loaded"]:
            _spur_duckdb_job(platform, name, "load")


class _SpurJevChannel:
    def __init__(self, descriptor):
        import socket
        self.socket = socket.socket(fileno=int(descriptor))
        self.socket.set_inheritable(False)
        self.lease = self._frame()

    def _read(self, count):
        chunks = bytearray()
        while len(chunks) < count:
            chunk = self.socket.recv(count - len(chunks))
            if not chunk:
                raise RuntimeError("Jev private credential channel closed")
            chunks.extend(chunk)
        return bytes(chunks)

    def _frame(self):
        return self._read(int.from_bytes(self._read(4), "big"))

    def _authorize(self, resolve):
        self.socket.sendall(len(self.lease).to_bytes(4, "big") + self.lease + bytes([resolve]))
        status = self._read(1)
        if status == b"\x02":
            return False
        if status != b"\x01":
            raise RuntimeError("denied")
        return True

    def available(self):
        try:
            return self._authorize(False)
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None

    def request(self):
        try:
            if not self._authorize(True):
                return None
            token = self._frame().decode("ascii")
            if not token or any(not 33 <= ord(c) <= 126 for c in token):
                raise ValueError("invalid credential")
            return token
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None


# Repeated generated SQL preambles preserve the captured channel. Failed
# attachment remains failed until kernel restart; never retry a legacy file.
if "_SPUR_JEV_BROKER" not in globals():
    _SPUR_JEV_BROKER = None
    _SPUR_JEV_BROKER_FAILED = False


def _spur_jev_channel():
    import os
    global _SPUR_JEV_BROKER, _SPUR_JEV_BROKER_FAILED
    if _SPUR_JEV_BROKER_FAILED:
        raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.")
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            if _SPUR_JEV_BROKER is not None:
                raise RuntimeError("duplicate channel")
            _SPUR_JEV_BROKER = _SpurJevChannel(descriptor)
        except Exception:
            _SPUR_JEV_BROKER_FAILED = True
            raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.") from None
    return _SPUR_JEV_BROKER


def _spur_close_jev_channel():
    """Disable/revoke/replacement is terminal for this inherited descriptor."""
    import os
    global _SPUR_JEV_BROKER_FAILED
    _SPUR_JEV_BROKER_FAILED = True
    if _SPUR_JEV_BROKER is not None:
        _SPUR_JEV_BROKER.socket.close()
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            os.close(int(descriptor))
        except (ValueError, OSError):
            pass


class _SpurJevSession:
    def __init__(self, db):
        self.db = db
        self.token = None
        self.previous = None

    def __enter__(self):
        import os
        _spur_duckdb_baseline()
        runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
        connection = _spur_managed_connection(self.db)
        if runtime is None or runtime.db is not connection:
            return self
        entry = runtime._entry("jev")
        if not (entry["desired_enabled"] and entry["loaded"] and entry["signature_verified"]
                and "jev" in runtime.loaded):
            return self
        try:
            channel = _spur_jev_channel()
            if channel is None or not channel.available():
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
                return self
        except Exception:
            # A dead/revoked channel cannot renew in-place. Do not block ordinary
            # SQL, read a legacy file or install/load anything as a fallback.
            entry["restart_required"] = True
            entry["activation_transport_restart"] = True
            entry["reason"] = "restart_required"
            return self
        endpoint = self.db.sql("SELECT current_setting('jev_api_url')").fetchall()[0][0]
        if endpoint != "https://api.typesafe.ai/v1/systemone":
            raise RuntimeError("Managed Jev credentials require the TypeSafe HTTPS endpoint. Reset jev_api_url before running this cell.")
        self.db.sql("RESET jev_api_key")
        # Resolve only after installation and exact endpoint validation. The
        # broker rechecks lease, enablement and generation on every request.
        try:
            token = channel.request()
        except Exception:
            entry["restart_required"] = True
            entry["reason"] = "restart_required"
            return self
        if token is None:  # disabled while setup was in flight
            return self
        self.previous = os.environ.get("TYPESAFE_API_KEY")
        self.token = token
        os.environ["TYPESAFE_API_KEY"] = token
        return self

    def __exit__(self, error_type, error, traceback):
        import os
        import traceback as traceback_module

        if self.token is None:
            return False
        token = self.token
        self.token = None
        if self.previous is None:
            os.environ.pop("TYPESAFE_API_KEY", None)
        else:
            os.environ["TYPESAFE_API_KEY"] = self.previous
        self.previous = None
        # A remote API can echo a header in an error. Suppress the original
        # traceback chain and remove the exact managed token from the diagnostic.
        if error is not None and token in "".join(traceback_module.format_exception(error_type, error, traceback)):
            message = str(error).replace(token, "[redacted]")
            token = None
            error = None
            traceback = None
            raise RuntimeError(message) from None
        return False

def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name):
            raise ValueError("Invalid managed DuckDB extension identity")
        if source not in ("core", "community"):
            raise ValueError(
                "Managed catalog setup supports only Core and Community repositories; "
                "other repositories require explicit setup in notebook SQL"
            )
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        # Same-user/kernel arbitrary code remains trusted; this is not a sandbox.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
        }
        version = connection.sql("SELECT version()").fetchall()[0][0]
        repository_aware = version == "v2.0.0" and source == "community"
        selected_requires_named_load = repository_aware and connection.sql(
            "SELECT current_setting('allow_unsigned_extensions')"
        ).fetchall()[0][0] is False
        if version == "v2.0.0":
            directories = connection.sql("SELECT current_setting('extension_directories')").fetchall()[0][0]
            if directories:
                config["extension_directories"] = directories
            repository_dir = connection.sql("SELECT current_setting('extension_repository_directory')").fetchall()[0][0]
            if repository_dir:
                config["extension_repository_directory"] = repository_dir
        else:
            config["home_directory"] = connection.sql("SELECT current_setting('home_directory')").fetchall()[0][0]
            directory = connection.sql("SELECT current_setting('extension_directory')").fetchall()[0][0]
            if directory:
                config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchall() != connection.sql(query).fetchall():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    if repository_aware:
                        verifier.execute(f"INSTALL {name} FROM {source}").fetchall()
                    else:
                        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                        verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                directory = Path(private.name)
                if repository_aware:
                    platform = verifier.sql("PRAGMA platform").fetchall()[0][0]
                    directory = directory / version / platform
                    directory.mkdir(parents=True)
                path = directory / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                if repository_aware:
                    info = Path(artifact[5] + ".info")
                    if info.is_file():
                        shutil.copyfile(info, Path(str(path) + ".info"))
                    # Only this isolated database changes its lookup directory.
                    # Named LOAD retains DuckDB 2.0's repository signing keys.
                    verifier.execute("SET extension_directories = ?", [[private.name]]).fetchall()
                    verifier.execute(f"LOAD {name} FROM {source}").fetchall()
                else:
                    verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            if hashlib.sha256(path.read_bytes()).digest() != digest:
                raise RuntimeError("Extension artifact changed during verification")
            if selected_requires_named_load:
                # A strict DuckDB 2.0 host also needs the repository identity.
                # It independently checks signing on the selected connection.
                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
                connection.execute(f"LOAD {name} FROM {source}").fetchall()
                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            else:
                connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()


_spur_duckdb_baseline()
# SPUR DuckDB readiness comm v1
def _spur_register_readiness_comm(verifier_source):
    """Answer host observations without starting another notebook cell."""
    def on_open(comm, message):
        try:
            requirements = message.get("content", {}).get("data", {}).get("requirements", [])
            connection = globals().get("_SPUR_DUCKDB_CONNECTION")
            scope = {"_SPUR_CATALOG_EXTENSION_PROOFS": globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})}
            exec(verifier_source, scope)
            ready = bool(connection is not None and scope["_spur_catalog_extensions_ready"](connection, requirements))
            data = {"protocol": "spur.duckdb.readiness.v1", "ready": ready, "connection_id": str(id(connection)) if connection is not None else None}
        except Exception:
            # The channel exists, but the current connection cannot be checked.
            data = {"protocol": "spur.duckdb.readiness.v1", "ready": False, "connection_id": None}
        comm.close(data=data)

    try:
        get_ipython().kernel.comm_manager.register_target("spur.duckdb.readiness.v1", on_open)
    except Exception:
        pass

_spur_register_readiness_comm("def _spur_restore_catalog_extensions(connection, requirements):\n    \"\"\"Restore optional membership without making unrelated SQL depend on its success.\"\"\"\n    import sys\n    failures = globals().setdefault(\"_SPUR_CATALOG_EXTENSION_FAILURES\", {})\n    errors = {}\n    for item in requirements:\n        name, source = item.get(\"packageName\", \"\"), item.get(\"source\", \"\")\n        key = (id(connection), name, source)\n        if _spur_catalog_extensions_ready(connection, [item]):\n            failures.pop(key, None)\n            continue\n        previous = failures.get(key)\n        if previous and previous[0] is connection:\n            errors[name] = previous[1]\n            continue\n        try:\n            _spur_prepare_catalog_extensions(connection, [item])\n        except Exception as error:\n            errors[name] = str(error)\n            failures[key] = (connection, str(error))\n            print(f\"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.\", file=sys.stderr)\n    return errors\n\n\ndef _spur_catalog_extensions_ready(connection, requirements):\n    \"\"\"Read current connection identity and proofs without installing or loading code.\"\"\"\n    import re\n    rows = connection.sql(\n        \"SELECT extension_name, installed, loaded, install_mode, installed_from, \"\n        \"install_path, aliases FROM duckdb_extensions()\"\n    ).fetchall()\n    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}\n    proofs = globals().get(\"_SPUR_CATALOG_EXTENSION_PROOFS\", {})\n    for item in requirements:\n        name, source = item.get(\"packageName\", \"\"), item.get(\"source\", \"\")\n        if not re.fullmatch(r\"[a-z][a-z0-9_]*\", name) or source not in (\"core\", \"community\"):\n            return False\n        row = inventory.get(name)\n        if not row or not row[2]:\n            return False\n        if row[3] == \"STATICALLY_LINKED\" and source == \"core\":\n            continue\n        proof = proofs.get((id(connection), row[0]))\n        if not proof or proof[0] is not connection or proof[1] != source:\n            return False\n    return True\n\n\ndef _spur_prepare_connection_extensions(statements):\n    \"\"\"Use the same verifier for host-generated connection INSTALL/LOAD pairs.\"\"\"\n    import re\n    requirements, loads = {}, set()\n    for statement in statements:\n        install = re.fullmatch(r\"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?\", statement)\n        load = re.fullmatch(r\"LOAD ([a-z][a-z0-9_]*)\", statement)\n        if install:\n            name, source = install.group(1), install.group(2) or \"core\"\n            if name in requirements and requirements[name] != source:\n                raise ValueError(\"Conflicting managed connection repositories\")\n            requirements[name] = source\n        elif load:\n            loads.add(load.group(1))\n        else:\n            raise ValueError(\"Invalid managed connection extension statement\")\n    if loads != set(requirements):\n        raise ValueError(\"Managed connection extension installation and load must match\")\n    connection = globals().get(\"_SPUR_DUCKDB_CONNECTION\")\n    if connection is None:\n        connection = duckdb.default_connection() if hasattr(duckdb, \"default_connection\") else duckdb\n    _spur_prepare_catalog_extensions(connection, [\n        {\"packageName\": name, \"source\": source} for name, source in requirements.items()\n    ])\n\n\ndef _spur_prepare_catalog_extensions(connection, requirements):\n    \"\"\"Prepare signed packages on the notebook's shared DuckDB connection.\"\"\"\n    import duckdb as engine\n    import hashlib\n    from pathlib import Path\n    import re\n    import shutil\n    import tempfile\n\n    checked = {}\n    for item in requirements:\n        name, source = item.get(\"packageName\", \"\"), item.get(\"source\", \"\")\n        if not re.fullmatch(r\"[a-z][a-z0-9_]*\", name):\n            raise ValueError(\"Invalid managed DuckDB extension identity\")\n        if source not in (\"core\", \"community\"):\n            raise ValueError(\n                \"Managed catalog setup supports only Core and Community repositories; \"\n                \"other repositories require explicit setup in notebook SQL\"\n            )\n        if name in checked and checked[name] != source:\n            raise ValueError(\"One package cannot use two repositories in the same runtime\")\n        checked[name] = source\n        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.\n        globals().get(\"_SPUR_CATALOG_EXTENSION_FAILURES\", {}).pop((id(connection), name, source), None)\n\n    def inventory(db):\n        rows = db.sql(\n            \"SELECT extension_name, installed, loaded, install_mode, installed_from, \"\n            \"install_path, aliases FROM duckdb_extensions()\"\n        ).fetchall()\n        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}\n\n    def matches(row, source):\n        origin = row[4] or \"\"\n        repository = \"extensions.duckdb.org\" if source == \"core\" else \"community-extensions.duckdb.org\"\n        return origin == source or origin.rstrip(\"/\") in (\n            \"http://\" + repository, \"https://\" + repository)\n\n    # Proofs belong to this exact connection, surviving redefinition on SQL replay.\n    # Retain private immutable copies for the lifetime of the kernel.\n    proofs = globals().setdefault(\"_SPUR_CATALOG_EXTENSION_PROOFS\", {})\n    for name, source in checked.items():\n        row = inventory(connection).get(name)\n        if row and row[3] == \"STATICALLY_LINKED\" and source == \"core\":\n            if not row[2]:\n                connection.load_extension(name)\n            continue\n        proof = proofs.get((id(connection), row[0] if row else name))\n        if row and row[2]:\n            # The required HTTPFS baseline has an existing signed verifier and\n            # frozen copy. Share that exact proof, never a bare loaded flag or\n            # repository label. This also supports already-running baseline code.\n            if proof is None and row[0] == \"httpfs\" and source == \"core\":\n                runtime = globals().get(\"_SPUR_DUCKDB_RUNTIME\")\n                if runtime is not None and runtime.db is connection and \"httpfs\" in runtime.loaded:\n                    baseline = runtime._entry(\"httpfs\")\n                    private = runtime.verified.get(\"httpfs\")\n                    if baseline.get(\"signature_verified\") is True and baseline.get(\"loaded\") is True and private is not None:\n                        try:\n                            digest = hashlib.sha256((Path(private.name) / \"httpfs.duckdb_extension\").read_bytes()).digest()\n                            if digest.hex() == baseline.get(\"artifact_digest\"):\n                                proof = (connection, source, private, digest)\n                                proofs[(id(connection), row[0])] = proof\n                        except OSError:\n                            pass\n            if proof and proof[0] is connection and proof[1] == source:\n                continue\n            raise RuntimeError(f\"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook\")\n\n        # The shared connection permits the app's bundled unsigned gateway.\n        # Always verify third-party bytes in an isolated engine with signing ON.\n        # Same-user/kernel arbitrary code remains trusted; this is not a sandbox.\n        config = {\n            \"allow_unsigned_extensions\": \"false\",\n            \"allow_extensions_metadata_mismatch\": \"false\",\n            \"allow_community_extensions\": \"true\" if source == \"community\" else \"false\",\n            \"autoload_known_extensions\": \"false\",\n            \"autoinstall_known_extensions\": \"false\",\n        }\n        version = connection.sql(\"SELECT version()\").fetchall()[0][0]\n        repository_aware = version == \"v2.0.0\" and source == \"community\"\n        selected_requires_named_load = repository_aware and connection.sql(\n            \"SELECT current_setting('allow_unsigned_extensions')\"\n        ).fetchall()[0][0] is False\n        if version == \"v2.0.0\":\n            directories = connection.sql(\"SELECT current_setting('extension_directories')\").fetchall()[0][0]\n            if directories:\n                config[\"extension_directories\"] = directories\n            repository_dir = connection.sql(\"SELECT current_setting('extension_repository_directory')\").fetchall()[0][0]\n            if repository_dir:\n                config[\"extension_repository_directory\"] = repository_dir\n        else:\n            config[\"home_directory\"] = connection.sql(\"SELECT current_setting('home_directory')\").fetchall()[0][0]\n            directory = connection.sql(\"SELECT current_setting('extension_directory')\").fetchall()[0][0]\n            if directory:\n                config[\"extension_directory\"] = directory\n        private = None\n        try:\n            with engine.connect(config=config) as verifier:\n                for query in (\"SELECT version()\", \"PRAGMA platform\"):\n                    if verifier.sql(query).fetchall() != connection.sql(query).fetchall():\n                        raise RuntimeError(\"DuckDB runtime changed during extension verification\")\n                artifact = inventory(verifier).get(name)\n                if not artifact or not artifact[1]:\n                    if repository_aware:\n                        verifier.execute(f\"INSTALL {name} FROM {source}\").fetchall()\n                    else:\n                        repository = \"extensions.duckdb.org\" if source == \"core\" else \"community-extensions.duckdb.org\"\n                        verifier.install_extension(name, repository_url=\"http://\" + repository)\n                    artifact = inventory(verifier).get(name)\n                if not artifact or not artifact[1]:\n                    raise RuntimeError(f\"{name} installation is unavailable for this DuckDB runtime\")\n                if artifact[3] != \"REPOSITORY\" or not matches(artifact, source):\n                    raise RuntimeError(f\"{name} is cached from another repository; resolve its source before retrying\")\n                private = tempfile.TemporaryDirectory(prefix=\"spur-duckdb-catalog-\")\n                directory = Path(private.name)\n                if repository_aware:\n                    platform = verifier.sql(\"PRAGMA platform\").fetchall()[0][0]\n                    directory = directory / version / platform\n                    directory.mkdir(parents=True)\n                path = directory / (artifact[0] + \".duckdb_extension\")\n                shutil.copyfile(artifact[5], path)\n                path.chmod(0o400)\n                digest = hashlib.sha256(path.read_bytes()).digest()\n                if repository_aware:\n                    info = Path(artifact[5] + \".info\")\n                    if info.is_file():\n                        shutil.copyfile(info, Path(str(path) + \".info\"))\n                    # Only this isolated database changes its lookup directory.\n                    # Named LOAD retains DuckDB 2.0's repository signing keys.\n                    verifier.execute(\"SET extension_directories = ?\", [[private.name]]).fetchall()\n                    verifier.execute(f\"LOAD {name} FROM {source}\").fetchall()\n                else:\n                    verifier.load_extension(str(path))  # signature and ABI checked by DuckDB\n                if hashlib.sha256(path.read_bytes()).digest() != digest:\n                    raise RuntimeError(\"Extension artifact changed during verification\")\n            snapshot = globals().get(\"_spur_snapshot_function_registration\")\n            before = snapshot(connection) if callable(snapshot) else None\n            if hashlib.sha256(path.read_bytes()).digest() != digest:\n                raise RuntimeError(\"Extension artifact changed during verification\")\n            if selected_requires_named_load:\n                # A strict DuckDB 2.0 host also needs the repository identity.\n                # It independently checks signing on the selected connection.\n                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:\n                    raise RuntimeError(\"Extension artifact changed during verification\")\n                connection.execute(f\"LOAD {name} FROM {source}\").fetchall()\n                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:\n                    raise RuntimeError(\"Extension artifact changed during verification\")\n            else:\n                connection.load_extension(str(path))  # load exactly the verified bytes\n            observed = inventory(connection).get(name)\n            if not observed or not observed[2]:\n                raise RuntimeError(f\"{name} did not load into the notebook connection\")\n            proofs[(id(connection), observed[0])] = (connection, source, private, digest)\n            record = globals().get(\"_spur_record_function_ownership\")\n            if callable(record):\n                record(connection, observed[0], before)\n            private = None\n        finally:\n            if private is not None:\n                private.cleanup()\n")
def _spur_reconcile_managed_datasources(desired_views, removed_views, desired_aliases, removed_aliases):
    # Only objects emitted by managed setup belong to this registry. Never drop
    # a whole schema: users may have created their own objects alongside ours.
    def key(value):
        return value.translate(str.maketrans('ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'))

    def ident(value):
        return '"' + value.replace('"', '""') + '"'

    def views(values):
        return {(key(schema), key(name)): (schema, name) for schema, name in values}

    wanted_views = views(desired_views)
    previous_views = views(globals().get('_SPUR_MANAGED_VIEWS', []))
    previous_views.update(views(removed_views))
    for identity, (schema, name) in previous_views.items():
        if identity not in wanted_views:
            duckdb.sql('DROP VIEW IF EXISTS "memory".' + ident(schema) + '.' + ident(name))

    wanted_aliases = {key(name) for name in desired_aliases}
    previous_aliases = set(globals().get('_SPUR_MANAGED_ALIASES', [])) | set(removed_aliases)
    attached = {key(row[0]): row[0] for row in duckdb.sql('SELECT database_name FROM duckdb_databases()').fetchall()}
    for name in previous_aliases:
        identity = key(name)
        if identity in wanted_aliases or identity in {'memory', 'temp', 'system'}:
            continue
        if identity in attached:
            duckdb.sql('DETACH ' + ident(attached[identity]))
        for registry_name in ('_SPUR_REMOTE_CONNECTIONS', '_SPUR_LAKEHOUSE_CONNECTIONS'):
            registry = globals().get(registry_name, {})
            for cached in list(registry):
                if key(cached) == identity:
                    del registry[cached]
        for cached in list(globals().get('_SPUR_REMOTE_ATTACHED', set())):
            if key(cached) == identity:
                _SPUR_REMOTE_ATTACHED.discard(cached)

    globals()['_SPUR_MANAGED_VIEWS'] = list(wanted_views.values())
    globals()['_SPUR_MANAGED_ALIASES'] = list(desired_aliases)

_spur_reconcile_managed_datasources([["github-pat","security_advisories"],["github-pat","authenticated_repos"],["github-pat","user_issues"],["github-pat","user_orgs"],["github-pat","user_gists"],["github-pat","starred"],["github-pat","notifications"],["github-pat","user_followers"],["github-pat","user_emails"],["github-pat","authenticated_user"],["github-pat","user_following"]], [["github-pat","security_advisories"],["github-pat","authenticated_repos"],["github-pat","user_issues"],["github-pat","user_orgs"],["github-pat","user_gists"],["github-pat","starred"],["github-pat","notifications"],["github-pat","user_followers"],["github-pat","user_emails"],["github-pat","authenticated_user"],["github-pat","user_following"]], [], [])
try:
    duckdb.sql("SET search_path = 'memory'")
except Exception:
    pass
if not globals().get("_SPUR_REST_EXTENSION_LOADED", False):
    if _SPUR_DUCKDB_GATEWAY_COMPATIBLE:
        try:
            duckdb.sql(f"LOAD '{_SPUR_DUCKDB_EXTENSION_SQL}'")
            _SPUR_REST_EXTENSION_LOADED = True
        except Exception as _spur_rest_load_err:
            _SPUR_REST_EXTENSION_LOADED = False
            print(
                f"SPUR: failed to LOAD spur_rest extension ({_SPUR_DUCKDB_EXTENSION_PATH}): {_spur_rest_load_err}",
                file=sys.stderr,
            )
_spur_live_api_tables = duckdb.sql("SELECT count(*) FROM duckdb_functions() WHERE function_name = 'spur_rest_table'").fetchall()[0][0] > 0
_spur_native_sources = set()
if duckdb.sql("SELECT count(*) FROM duckdb_functions() WHERE function_name = 'spur_native_refresh'").fetchall()[0][0]:
    for _spur_source in ["github","github-pat"]:
        for _spur_owner, _spur_sql, _spur_revision, _spur_error in duckdb.execute("SELECT * FROM spur_native_refresh(?)", [_spur_source]).fetchall():
            _spur_native_sources.update((_spur_source, _spur_owner))
            if _spur_error:
                print(f"SPUR: native source {_spur_owner} refresh failed: {_spur_error}", file=sys.stderr)
                continue
            try:
                duckdb.execute(_spur_sql)
            except Exception as _spur_refresh_error:
                duckdb.execute("ROLLBACK")
                print(f"SPUR: native source {_spur_owner} publication failed: {_spur_refresh_error}", file=sys.stderr)
try:
    _SPUR_REST_TF_NAMES = {
        r[0]
        for r in duckdb.sql(
            "SELECT DISTINCT function_name FROM duckdb_functions() WHERE function_name LIKE '%\\_%' ESCAPE '\\'"
        ).fetchall()
    }
except Exception as _spur_tf_snap_err:
    _SPUR_REST_TF_NAMES = set()
    print(
        f"SPUR: could not snapshot table functions: {_spur_tf_snap_err}",
        file=sys.stderr,
    )
duckdb.sql("CREATE SCHEMA IF NOT EXISTS \"github-pat\"") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"security_advisories\" AS SELECT * FROM spur_rest_table('github', 'security_advisories')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"security_advisories\" AS SELECT * FROM \"github_security_advisories\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_repos\" AS SELECT * FROM spur_rest_table('github', 'authenticated_repos')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_repos\" AS SELECT * FROM \"github_authenticated_repos\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_issues\" AS SELECT * FROM spur_rest_table('github', 'user_issues')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_issues\" AS SELECT * FROM \"github_user_issues\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_orgs\" AS SELECT * FROM spur_rest_table('github', 'user_orgs')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_orgs\" AS SELECT * FROM \"github_user_orgs\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_gists\" AS SELECT * FROM spur_rest_table('github', 'user_gists')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_gists\" AS SELECT * FROM \"github_user_gists\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"starred\" AS SELECT * FROM spur_rest_table('github', 'starred')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"starred\" AS SELECT * FROM \"github_starred\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"notifications\" AS SELECT * FROM spur_rest_table('github', 'notifications')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"notifications\" AS SELECT * FROM \"github_notifications\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_followers\" AS SELECT * FROM spur_rest_table('github', 'user_followers')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_followers\" AS SELECT * FROM \"github_user_followers\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_emails\" AS SELECT * FROM spur_rest_table('github', 'user_emails')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_emails\" AS SELECT * FROM \"github_user_emails\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_user\" AS SELECT * FROM spur_rest_table('github', 'authenticated_user')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_user\" AS SELECT * FROM \"github_authenticated_user\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_following\" AS SELECT * FROM spur_rest_table('github', 'user_following')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_following\" AS SELECT * FROM \"github_user_following\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None


In [4]:
-- R6: narrow live read through the existing saved connection.
SELECT current_timestamp AS collected_at, number,title,state,user_login,html_url,updated_at
FROM github_search_issues("q"='repo:duckdb/duckdb is:pr 28731',"sort"='updated',"order"='desc')
ORDER BY updated_at DESC LIMIT 5;

collected_at,number,title,state,user_login,html_url,updated_at
2026-11-18 14:27:12.609839+0700,28731,Preserve aggregate domains for proven singleton groups,open,kevintruong,https://github.com/duckdb/duckdb/pull/28731,2026-11-17T15:22:12Z


In [7]:
# R6: local Git and history source surface on the same named 2.0 engine.
import subprocess
repository=Path('/Users/kevintruong/project/duckdb')
def git_read(*args):
 return subprocess.run(['git','-C',str(repository),*args],check=True,capture_output=True,text=True).stdout
tracked_head=git_read('ls-tree','-r','--full-tree',review_data['head']).splitlines()
changed_rows=git_read('diff','--name-status',review_data['base'],review_data['head']).splitlines()
repo_probe={'repository':str(repository),'base':review_data['base'],'head':review_data['head'],
 'tracked_head_entries':len(tracked_head),'changed_files':changed_rows,
 'history_functions':engine.sql("SELECT function_name,parameters,parameter_types FROM duckdb_functions() WHERE function_name IN ('git_log','git_blame','git_diff') ORDER BY function_name").fetchall()}
assert len(tracked_head)==18342 and len(changed_rows)==5
print(json.dumps(repo_probe,indent=2))

{
  "repository": "/Users/kevintruong/project/duckdb",
  "base": "629e395a854d72dcb1f414e1a0451c68ac0e9f1f",
  "head": "d29b26bb92ac940a0807be0f43cc385b31623839",
  "tracked_head_entries": 18342,
  "changed_files": [
    "M\tsrc/function/aggregate/distributive/first_last_any.cpp",
    "M\tsrc/optimizer/relation_statistics/relation_statistics_helper.cpp",
    "M\ttest/configs/stats_dependent_tests.json",
    "M\ttest/configs/verify_aggregate_state_export.json",
    "A\ttest/sql/optimizer/statistics/distinct_on_stats_propagation.test"
  ],
  "history_functions": [
    [
      "git_blame",
      [
        "col0",
        "repo_path",
        "revision",
        "min_line",
        "max_line",
        "ignore_whitespace",
        "use_mailmap",
        "first_parent"
      ],
      [
        "VARCHAR",
        "VARCHAR",
        "VARCHAR",
        "BIGINT",
        "BIGINT",
        "BOOLEAN",
        "BOOLEAN",
        "BOOLEAN"
      ]
    ],
    [
      "git_log",
      [
        "repo_

In [8]:
# R6: host-agent receipt is explicit; the named cell does not secretly fetch credentials.
live_pr_receipt={"source":"GitHub connector get_pr_info","number":28731,"head_sha":"d29b26bb92ac940a0807be0f43cc385b31623839","base_sha":"629e395a854d72dcb1f414e1a0451c68ac0e9f1f","updated_at":"2026-11-17T15:22:12Z","state":"open","requested_reviewers":None,"requested_team_reviewers":None}
assert live_pr_receipt['head_sha']==review_data['head'] and live_pr_receipt['base_sha']==review_data['base']
history_rollups=[]
for path in [f['path'] for f in review_data['files'] if f['path'].endswith('.cpp')]:
 uri=engine.execute('SELECT git_uri(?,?,?)',[str(repository),path,review_data['base']]).fetchall()[0][0]
 history=engine.sql('SELECT author_name,count(*) AS commits,max(author_date)::VARCHAR AS latest FROM git_log(?) GROUP BY author_name ORDER BY commits DESC,author_name',params=[uri]).fetchall()
 history_rollups.append({'file_path':path,'base_revision':review_data['base'],'authors':history})
collector_observation={'pr_receipt':live_pr_receipt,'tracked_head_entries':len(tracked_head),'changed_files':len(changed_rows),'history':history_rollups,'inbox_source':'Separate managed SQL cell using exact saved github-pat definition','automatic_named_pipeline':False,'current_reviewers':'unknown: connector returned null, not a verified empty assignment list'}
print(json.dumps(collector_observation,indent=2))

{
  "pr_receipt": {
    "source": "GitHub connector get_pr_info",
    "number": 28731,
    "head_sha": "d29b26bb92ac940a0807be0f43cc385b31623839",
    "base_sha": "629e395a854d72dcb1f414e1a0451c68ac0e9f1f",
    "updated_at": "2026-11-17T15:22:12Z",
    "state": "open",
    "requested_reviewers": null,
    "requested_team_reviewers": null
  },
  "tracked_head_entries": 18342,
  "changed_files": 5,
  "history": [
    {
      "file_path": "src/optimizer/relation_statistics/relation_statistics_helper.cpp",
      "base_revision": "629e395a854d72dcb1f414e1a0451c68ac0e9f1f",
      "authors": [
        [
          "Laurens Kuiper",
          22,
          "2026-11-13 10:09:06"
        ],
        [
          "Robinho-MR",
          3,
          "2026-11-05 20:22:47"
        ],
        [
          "Max Gabrielsson",
          1,
          "2026-08-24 08:06:29"
        ],
        [
          "Shubham Chaurasia",
          1,
          "2026-11-02 08:23:00"
        ],
        [
          "xproblem

In [ ]:
// R1: this must execute through the saved named target, never a standalone fallback.
const cfg = await import('@review/review_config.ts');
console.log(JSON.stringify({label:cfg.label,deno:Deno.version,pid:Deno.pid,cwd:Deno.cwd(),ports_api:spur.api_version}));

## R2 authoring admission result

A named `review-engine` Python cell declaring an Arrow producer was rejected atomically by `notebook_insert_cell` with `ports_unavailable: named Python profiles require a qualified host-fenced publication bridge`. No cell or port was created. This confirms the gate exists before execution as well as in the importable `spur` stub; adding DAG metadata cannot enable publication.

The R3 code below states the future asynchronous consumer contract. On this build it is blocked at named Deno startup, so it establishes no successful port read.

In [ ]:
// R3: acceptance probe; no producer is qualified or silently substituted.
if (spur.api_version !== 2) throw Error('Expected project ports v2');
const table = await spur.get('review_run_probe');
console.log(JSON.stringify({rows:table.numRows,ports_api:spur.api_version}));

In [9]:
# Current source audit; raw source stays local and lifecycle guards remain unchanged.
source_repo=Path('/Users/kevintruong/project/spur-notebook')
source_ranges={
 'src/dag/engine.rs':[(351,366)],
 'src/mcp/tools/native_profile_lifecycle.rs':[(18,31),(136,145),(180,189)],
 'jute-notebook/src-tauri/src/kernel_config/target.rs':[(220,236)],
 'jute-notebook/src-tauri/src/kernel_config/runtime_provenance.rs':[(180,197)],
 'src/dag/deno_publication_bridge.rs':[(256,269)],
 'jute-notebook/src-tauri/src/assets/deno/ports_project_v2.js':[(219,232)]}
source_audit={'head':subprocess.run(['git','-C',str(source_repo),'rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip(),'files':[]}
for path,ranges in source_ranges.items():
 raw=(source_repo/path).read_bytes(); lines=raw.decode().splitlines()
 source_audit['files'].append({'path':path,'sha256':hashlib.sha256(raw).hexdigest(),'excerpts':[{'start':lo,'end':hi,'text':'\n'.join(lines[lo-1:hi])} for lo,hi in ranges]})
print(json.dumps(source_audit,indent=2))

{
  "head": "c686dcf6ce81d1cc517cd11f0db0edc7062ef9b7",
  "files": [
    {
      "path": "src/dag/engine.rs",
      "sha256": "3cf79ef72af674c326ec95897c21a53987046423b4246cf305e00830fa379d43",
      "excerpts": [
        {
          "start": 351,
          "end": 366,
          "text": "    fn ensure_kernel<'a>(\n        &'a self,\n        request: KernelEnsureRequest,\n    ) -> Pin<Box<dyn Future<Output = Result<(), EngineError>> + Send + 'a>> {\n        Box::pin(async move {\n            // Native kernels launch lazily at guarded cell dispatch, where the\n            // canonical target is available to bind the immutable slot owner.\n            if matches!(\n                code_type_for_spec(&request.spec_name),\n                Some(CodeType::Cpp | CodeType::Rust)\n            ) || (request.spec_name == \"python3\"\n                && request.slot_id.starts_with(\"native-profile:\"))\n            {\n                return Ok(());\n            }\n            if let Some(state) = s

In [10]:
# Final native and evidence assertions on this exact named connection.
cpp_symbols=engine.sql("SELECT spur_graph_symbols('probe.cpp','int helper() { return 1; } int caller() { return helper(); }')").fetchall()[0][0]
refs=[row['evidence'] for context in review_data['contexts'].values() for row in context['rows']]
assert len(cpp_symbols)==2 and all(ref in review_data['evidence'] for ref in refs)
print(json.dumps({'cpp_symbols':len(cpp_symbols),'context_row_refs':len(refs),'all_refs_exist':True,'grammar':engine.sql("SELECT current_setting('active_grammar_extensions')").fetchall()[0][0],'fixture_sha256':hashlib.sha256(review_fixture.read_bytes()).hexdigest()},indent=2))

{
  "cpp_symbols": 2,
  "context_row_refs": 12,
  "all_refs_exist": true,
  "grammar": [
    "duckpgq"
  ],
  "fixture_sha256": "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2"
}


## Outcome and repair boundary

**R4 is fixed in this experiment:** `SET active_grammar_extensions = ['duckpgq']` enables real property-graph queries on the named DuckDB 2.0 connection. Three context paths and one syntax-only path match recursive SQL; removing the setting reproduces failure.

**R5 is qualified for the reference package:** a separate app notebook embeds a valid manifest, declares its frontend and active-script capability, renders the existing captured review interface, passes a clean cascade and full doctor at Level 2. It has no port pipeline or MCP server. Browser interaction/permission grants were not newly tested, and no app was packed.

**R1 is a current-source routing defect:** cold named Deno is missing from both MCP lazy ensure and profile lifecycle resolution. The repair must preserve factory Deno and authoritative UUID/notebook/spec guards; blindly enumerating factory Deno can panic because its native target is intentionally absent.

**R2 needs a qualified Python bridge:** both module calls and authoring a named producer reject publication. The two-cell origin experiment shows that the old async task retains cell A while a plain thread can report cell B. Never use latest parent identity alone to authorize publication. Preserve process, run, source/version, selected profile and lifecycle fences before lifting the gate.

**R3 depends on R1/R2:** its future contract uses `await spur.get(...)`. Current named Deno v2 does not provide the standalone `spur.anywidget()` helper, so stateful frontend composition needs a separate acceptance check. The consumer probe never executed.

**R6 has verified stages:** the saved GitHub connection returned PR #28731, connector metadata matches the pinned base/head, named Python enumerated all 18,342 tracked head entries and DuckTails queried real base-pinned history. The automatic collector/publication orchestration remains unimplemented. Null reviewer fields are unknown. Reattaching the connection dropped metadata; original provider/reference were restored through supported host commands and read back successfully, without changing secret values.

Solve now passes **9/13 supplied qualification bindings**; R1, R2, R3 and automatic R6 remain unqualified. Its origin consistency rule passes the async observation and rejects using the plain thread's latest parent as its spawning-cell identity. These results verify encoded evidence and policy, not the unimplemented runtime.

The implementation plan separates the narrow Deno routing fix, Python origin/bridge qualification, Deno consumer/frontend composition, and coherent collector orchestration. Current-source hashes and concrete regression witnesses are retained with this lab.